In [1]:
!pip install unsloth
!pip install -U bitsandbytes einops wandb datasets peft trl accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 137.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
!pip freeze > requirements.txt

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=max_seq_length,
    dtype=torch.float16,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRM

In [3]:
messages = [
    {"role": "user", "content": "Explain artificial intelligence in simple words"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [4]:
def generate_response(prompt, temperature=0.7):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        use_cache=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
result = generate_response(prompt, temperature=0.1)
print(result)

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

**What is Artificial Intelligence (AI)?**

Artificial Intelligence (AI) is a way to make computers and machines think and act like humans. It's like teaching a computer to learn from experience, make decisions, and solve problems on its own.

**How does AI work?**

Imagine you're trying to teach a child to recognize pictures of animals. You show them many pictures of cats and dogs, and they learn to identify them. AI works in a similar way. It's trained on a large amount of data, like pictures, words, or numbers, and it learns to recognize patterns and make decisions based on that data.

**Types of AI:**

1. **Narrow or Weak AI:** This type of AI is designed to perform a specific task, like playing chess or recognizing faces.
2. **General or Strong AI:** This type of AI is like a super-smart computer that can think and learn like a human being.
3. **Sup

In [6]:
result = generate_response(prompt, temperature=0.2)
print(result)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

**What is Artificial Intelligence (AI)?**

Artificial Intelligence (AI) is a way to make machines think and act like humans. It's like giving a computer the ability to learn, reason, and solve problems on its own.

**How does AI work?**

Imagine you're teaching a child to recognize pictures of cats and dogs. You show them many pictures and say, "This is a cat" or "This is a dog." Over time, the child learns to recognize the differences between the two. AI works in a similar way. It's trained on a lot of data (like pictures, words, or numbers) and learns to recognize patterns and make decisions based on that data.

**Types of AI:**

1. **Narrow or Weak AI:** This type of AI is designed to perform a specific task, like playing chess or recognizing faces.
2. **General or Strong AI:** This type of AI is like a super-smart computer that can do anything a hum

In [7]:
result = generate_response(prompt, temperature=0.3)
print(result)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

**What is Artificial Intelligence (AI)?**

Artificial Intelligence (AI) is a way to make computers think and act like humans. It's like teaching a machine to learn from experience, make decisions, and solve problems on its own.

**How does AI work?**

Imagine you're teaching a child to recognize pictures of animals. You show the child many pictures of cats and dogs, and they learn to tell the difference between them. AI works in a similar way, but with computers and data.

Here's a simple example:

1. **Data collection**: We gather a lot of information about something, like pictures of cats and dogs.
2. **Learning**: A computer algorithm (a set of instructions) looks at the data and tries to find patterns and relationships between the pictures.
3. **Pattern recognition**: The algorithm learns to recognize the differences between cats and dogs.
4. **Deci

In [8]:
result = generate_response(prompt, temperature=0.4)
print(result)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

**What is Artificial Intelligence (AI)?**

Artificial Intelligence (AI) is a type of computer programming that allows machines to think and act like humans. It's like a super smart robot that can learn, reason, and solve problems on its own.

**How does AI work?**

Imagine you're teaching a child to recognize pictures of cats and dogs. You show them many pictures and say "this is a cat" or "this is a dog." Over time, the child learns to recognize the differences between the two. AI works in a similar way, but instead of a child, it's a computer program that learns from data and examples.

**Types of AI:**

1. **Narrow or Weak AI:** This type of AI is designed to perform a specific task, like a calculator or a language translation app.
2. **General or Strong AI:** This type of AI is like a super-smart robot that can learn and reason like a human. It's st

In [9]:
result = generate_response(prompt, temperature=0.5)
print(result)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

Artificial intelligence (AI) is a way to make machines think and act like humans. It's like giving a computer a brain so it can do things on its own, like:

1. **Learning**: AI can learn from data and experiences, just like a human learns from school or life.
2. **Reasoning**: AI can make decisions based on what it has learned, like a human would make a choice.
3. **Problem-solving**: AI can find solutions to problems, like a human would figure out a puzzle.

AI is used in many things, such as:

1. **Virtual assistants**: Like Siri, Alexa, or Google Assistant, which can help with tasks and answer questions.
2. **Self-driving cars**: Cars that can drive themselves, using cameras and sensors to navigate roads.
3. **Image recognition**: AI can recognize objects, people, or animals in pictures or videos.
4. **Chatbots**: AI-powered chatbots can talk to huma

In [10]:
result = generate_response(prompt, temperature=0.6)
print(result)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

Artificial Intelligence (AI) is a way to make machines think and act like humans. It's like teaching a computer to learn from data, make decisions, and solve problems on its own.

Imagine you have a super smart robot that can:

1. **Learn**: It can read, understand, and remember information from books, articles, and experiences.
2. **Reason**: It can think logically and make decisions based on what it's learned.
3. **Imitate**: It can mimic human behavior, like recognizing faces, understanding language, and even creating art.

There are many types of AI, such as:

1. **Machine Learning**: This is like teaching a child to ride a bike. You show it how to balance and steer, and it learns to do it on its own.
2. **Natural Language Processing (NLP)**: This is like having a conversation with a chatbot. It understands what you say and responds accordingly.
3. 

In [11]:
result = generate_response(prompt, temperature=0.7)
print(result)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

Artificial intelligence (AI) is a technology that helps machines or computers think and act like humans. It's like teaching a computer to learn from experience, make decisions, and solve problems on its own.

Imagine you're playing a game like chess. A human player would think about the moves, consider the consequences, and make a decision. AI is similar, but instead of a human brain, it uses a computer program to make decisions.

There are several types of AI:

1. **Narrow or Weak AI**: This type of AI is designed to perform a specific task, like a calculator or a self-driving car.
2. **General or Strong AI**: This type of AI is like a super-smart human, able to learn and do anything a human can do.
3. **Superintelligence**: This type of AI is far more intelligent than humans, able to solve complex problems that humans can't even understand.

AI uses v

In [12]:
result = generate_response(prompt, temperature=0.8)
print(result)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Explain artificial intelligence in simple wordsassistant

Artificial intelligence (AI) is a type of computer technology that allows machines to think and learn like humans. It's like a super smart robot that can do many things on its own, without being programmed exactly what to do.

Imagine you have a friend who is very good at solving puzzles and playing games. They can look at a picture and recognize what it is, or read a book and understand what it's about. They can even learn from their mistakes and get better over time.

That's basically what AI is – a computer program that can do things like:

1. **Learn from data**: AI can look at lots of information and figure out patterns and rules.
2. **Make decisions**: AI can use what it's learned to make decisions and take actions.
3. **Improve over time**: AI can get better at things as it gets more experience.

There are many types of AI, including:

1. **Virtu